---
<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Dinámica del clima   
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:Blue; font-size:160%;">Unidad 2:Sistemas monzónicos</span><br/>
  <span style="color:Blue; font-size:140%;">Facultad de Ciencias  |  Semestre 2027-I</span><br/>
</p>

---

# **<font color="Navy">  Monzón de Asia y Australia </font>**
---
---

## 0. Preparación del entorno

In [ ]:
#pip install xarray cartopy pooch

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature

xr.set_options(keep_attrs=True)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

print('xarray', xr.__version__)

# **<font color="Navy">  1. Fundamentos físicos del monzón </font>**

### <font color="Navy"> 1.1 ¿Qué es un monzón?</font>

La palabra *monzón* (del árabe *mawsim*, "estación") describe originalmente la reversión estacional de los vientos sobre el océano Índico y el sur de Asia, aprovechada durante siglos por la navegación comercial. En climatología moderna, un **monzón** se define de forma más general como:

> Un sistema de circulación atmosférica regional en el que el viento predominante en niveles bajos **invierte su dirección entre el verano y el invierno**, asociado a un cambio igualmente marcado en la precipitación (estación húmeda vs. estación seca).

El sistema monzónico más importantes del planeta es el **monzón de Asia-Australia**  por ser el más extenso e intenso.
### <font color="Navy">1.2 Mecanismos físicos</font>

**(a) Contraste térmico tierra-océano.** Los continentes tienen una capacidad calorífica mucho menor que el océano. En verano el continente se calienta más rápido que el mar adyacente, generando una **baja térmica** en superficie (p. ej. la depresión térmica del noroeste de India/Pakistán, o la "baja de Australia del norte" en el verano austral). En invierno ocurre lo contrario: el continente se enfría más rápido, favoreciendo una **alta térmica** (p. ej. la alta de Siberia). Este contraste de presión impulsa un flujo de aire desde el océano (relativamente cálido y húmedo en verano) hacia el continente.

**(b) Viento térmico y migración de la ZCIT.** La Zona de Convergencia Intertropical (ZCIT) sigue, con cierto retraso, el máximo de calentamiento solar y por tanto migra estacionalmente hacia el hemisferio de verano. Sobre los continentes monzónicos esta migración es mucho más amplia que sobre el océano abierto, porque el calentamiento continental "atrae" a la ZCIT tierra adentro (p. ej. hasta ~25-30°N sobre India en julio-agosto).

**(c) Circulación de Hadley regional y flujo transecuatorial.** El monzón puede describirse como una **célula de Hadley regional** con ascenso sobre el continente cálido de verano y subsidencia sobre el océano o el continente frío del hemisferio opuesto. Esto obliga a un **flujo de niveles bajos que cruza el ecuador**, que se desvía por el efecto de Coriolis y da lugar a chorros de bajo nivel característicos, como el **chorro de Somalia** (Findlater jet) en el monzón de verano asiático.

**(d) Cizalladura vertical del viento zonal.** Una firma dinámica muy usada para cuantificar la intensidad del monzón es la diferencia entre el viento zonal en 850 hPa y en 200 hPa. Durante el monzón de verano, el flujo de niveles bajos es del oeste (viento del suroeste que trae humedad) mientras que en niveles altos domina un flujo del este (viento fácil tropical de este, *Tropical Easterly Jet*) sobre el anticiclón tibetano. Esta cizalladura fuerte y de signo opuesto entre niveles es la base del **índice de Webster–Yang** (Webster & Yang, 1992), que usaremos más adelante con datos reales.

### <font color="Navy">1.3 El "monzón global" como sistema acoplado</font>

Trenberth et al. (2000) y, más tarde, Wang & Ding (2008), mostraron que si uno mira la circulación divergente tropical en su conjunto, los distintos "monzones regionales" no son sistemas independientes sino **manifestaciones locales de un único sistema estacional planetario**: cuando el monzón de verano está en su apogeo en un hemisferio (p. ej. el monzón de verano asiático en julio-agosto), el sistema opuesto en el otro hemisferio está en su fase seca (el monzón australiano está en receso), y viceversa en enero-febrero. El aire que sale de un continente en su estación seca a menudo **alimenta directamente**, vía flujo transecuatorial, la convección del continente en su estación húmeda al otro lado del ecuador. Esta idea de "sube-y-baja" (*seesaw*) hemisférico es central para entender por qué el monzón asiático de invierno boreal (viento del noreste sobre el sudeste asiático) está física y dinámicamente conectado con el monzón australiano de verano austral (viento del noroeste sobre el norte de Australia): son, en gran medida, **la misma corriente de aire**, vista desde dos lados del ecuador.

Con esta idea en mente, en las secciones siguientes vamos a "ver" estos mecanismos en datos reales.



## <font color="Navy"> 2. Datos: cómo obtener ERA5 real (y qué usamos aquí en su lugar)</font>

### <font color="Navy"> 2.1 ERA5 real vía Copernicus Climate Data Store (CDS)</font>

ERA5 es el reanálisis atmosférico de quinta generación del ECMWF (Hersbach et al., 2020), con resolución horizontal de 0.25° x 0.25°, 137 niveles verticales, y cobertura desde 1940 hasta el presente, actualizado casi en tiempo real. Es, hoy en día, el estándar de facto para climatología y dinámica atmosférica observacional.

Para descargar ERA5 real necesitas:

1. Crear una cuenta gratuita en <https://cds.climate.copernicus.eu/>.
2. Aceptar la licencia del dataset que quieras usar (p. ej. *ERA5 monthly averaged data on pressure levels*).
3. Instalar el cliente `cdsapi` (`pip install cdsapi`) y guardar tu API key en `~/.cdsapirc`.

El siguiente bloque de código es **funcional y está listo para ejecutarse** en un entorno con acceso a internet y credenciales configuradas — descarga exactamente las variables que usaremos en este notebook (viento zonal, viento meridional y geopotencial en 850/500/200 hPa), pero como climatología mensual completa 1991-2020 en vez de solo dos meses:

```python
import cdsapi

c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-pressure-levels-monthly-means',
    {
        'product_type': 'monthly_averaged_reanalysis',
        'variable': ['u_component_of_wind', 'v_component_of_wind', 'geopotential'],
        'pressure_level': ['200', '500', '850'],
        'year': [str(y) for y in range(1991, 2021)],
        'month': [f'{m:02d}' for m in range(1, 13)],
        'time': '00:00',
        'area': [40, 30, -40, 160],   # Norte, Oeste, Sur, Este -> dominio Asia-Australia
        'format': 'netcdf',
    },
    'era5_uvz_asia_australia_1991_2020.nc'
)

# Para precipitación (esencial para monzones) el dataset relevante es:
c.retrieve(
    'reanalysis-era5-single-levels-monthly-means',
    {
        'product_type': 'monthly_averaged_reanalysis',
        'variable': 'total_precipitation',
        'year': [str(y) for y in range(1991, 2021)],
        'month': [f'{m:02d}' for m in range(1, 13)],
        'time': '00:00',
        'area': [40, 30, -40, 160],
        'format': 'netcdf',
    },
    'era5_precip_asia_australia_1991_2020.nc'
)
```

Con ese archivo, todo el análisis de este notebook (Secciones 3-5) se reproduce **sin cambiar la lógica del código**, solo cambiando `ds = xr.tutorial.open_dataset('eraint_uvz')` por `ds = xr.open_dataset('era5_uvz_asia_australia_1991_2020.nc')` y ajustando nombres de variables/dimensiones si difieren (ERA5 vía CDS suele usar `longitude`, `latitude`, `level`/`pressure_level`, `time`).

###  <font color="Navy"> Datos realmente usados en este notebook </font>

Usamos un dataset **real** del reanálisis **ERA-Interim** del ECMWF —el predecesor directo de ERA5, mismo centro, física de modelo muy similar y completamente adecuado para fines didácticos de dinámica de gran escala—, distribuido públicamente como parte de los datos de ejemplo de `xarray`. Contiene climatologías mensuales de **enero** (representativo del invierno boreal / verano austral) y **julio** (representativo del verano boreal / invierno austral) para:

- `u`: viento zonal (m/s)
- `v`: viento meridional (m/s)
- `z`: geopotencial (m² s⁻²)

en los niveles de presión **850, 500 y 200 hPa**, en todo el globo, con resolución de ~0.75°.

Esto es precisamente lo que necesitamos: los dos meses que mejor representan los extremos opuestos del ciclo monzónico.


In [ ]:
# Paso 1) subir el conjunto de datos
# Dataset real de reanálisis ERA-Interim (ECMWF), climatología mensual enero/julio
ds = xr.tutorial.open_dataset('eraint_uvz')
ds



Observa que la dimensión `month` toma los valores `1` (enero) y `7` (julio), `level` toma `200, 500, 850` hPa, y `latitude`/`longitude` cubren el globo completo en una malla regular de 0.75°. A partir de aquí, `1` = enero y `7` = julio en todas las selecciones `.sel(month=...)`.



## <font color="Navy"> 3. El monzón asiático </font>

Vamos a construir una función auxiliar de graficado que reutilizaremos para todos los mapas de viento (velocidad sombreada + vectores), y otra para mapas de cizalladura vertical. Definir estas funciones una sola vez nos permite comparar enero vs. julio, y Asia vs. Australia, con un código consistente.


In [ ]:

MONTH_NAMES = {1: 'Enero (DEF, verano austral / invierno boreal)',
               7: 'Julio (JJA, verano boreal / invierno austral)'}

def plot_wind_panel(ax, ds, month, level, extent, skip=3, vmax=20, cmap='viridis'):
    '''Dibuja velocidad del viento (sombreado) + vectores u,v en un nivel y mes dados.'''
    d = ds.sel(month=month, level=level)
    speed = np.sqrt(d.u**2 + d.v**2)

    cf = ax.contourf(d.longitude, d.latitude, speed,
                      levels=np.linspace(0, vmax, 11), cmap=cmap,
                      transform=ccrs.PlateCarree(), extend='max')
    ax.quiver(d.longitude.values[::skip], d.latitude.values[::skip],
              d.u.values[::skip, ::skip], d.v.values[::skip, ::skip],
              transform=ccrs.PlateCarree(), scale=420, width=0.0022, color='k', alpha=0.75)

    ax.coastlines(resolution='110m', linewidth=0.7)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, color='gray')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 7}
    gl.ylabel_style = {'size': 7}
    ax.set_title(f'{level} hPa — {MONTH_NAMES[month]}', fontsize=10)
    return cf
ASIA_EXTENT = [30, 140, -20, 40]   # lon_min, lon_max, lat_min, lat_max



### <font color="Navy">  Reversión del viento en niveles bajos (850 hPa) <font >

La firma más clásica del monzón asiático es la reversión del viento cerca de la superficie sobre el océano Índico y el sur/sureste de Asia:

- En **enero**, predominan los vientos del **noreste** (monzón de invierno / "monzón seco"), que fluyen desde el continente asiático frío hacia el océano.
- En **julio**, predominan los vientos del **suroeste** (monzón de verano / "monzón húmedo"), que transportan enormes cantidades de humedad desde el océano Índico hacia el subcontinente indio y el sureste asiático.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5),
                          subplot_kw={'projection': ccrs.PlateCarree(central_longitude=90)})

cf1 = plot_wind_panel(axes[0], ds, month=1, level=850, extent=ASIA_EXTENT, vmax=18)
cf2 = plot_wind_panel(axes[1], ds, month=7, level=850, extent=ASIA_EXTENT, vmax=18)

cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', shrink=0.6, pad=0.08)
cbar.set_label('Velocidad del viento en 850 hPa (m/s)')
fig.suptitle('Monzón asiático: viento en niveles bajos (850 hPa)', y=1.03, fontsize=13)
plt.show()



**Lo que hay que observar:**

- En julio aparece un flujo intenso y coherente del suroeste que cruza el ecuador frente a la costa de África oriental/Somalia y gira hacia el subcontinente indio: es el **chorro de bajo nivel de Somalia** (*Findlater jet*), uno de los transportadores de humedad más importantes del planeta.
- En enero ese mismo sector muestra vientos mucho más débiles y de dirección opuesta (componente del noreste sobre el Golfo de Bengala y el sureste asiático).

Cuantifiquemos el chorro de Somalia con los propios datos, promediando en una caja frente a la costa de Somalia (0-10°N, 40-55°E):


In [ ]:

somali_box = ds.sel(longitude=slice(40, 55), latitude=slice(10, 0), level=850)
u_somali = somali_box.u.mean(dim=['longitude', 'latitude'])
v_somali = somali_box.v.mean(dim=['longitude', 'latitude'])
speed_somali = np.sqrt(u_somali**2 + v_somali**2)

for m in [1, 7]:
    print(f"{MONTH_NAMES[m]}: u={u_somali.sel(month=m).item():+.2f} m/s, "
          f"v={v_somali.sel(month=m).item():+.2f} m/s, "
          f"|V|={speed_somali.sel(month=m).item():.2f} m/s")



Con los datos reales de ERA-Interim, en esta caja frente a Somalia el viento pasa de ser débil y del noreste en enero (~4-5 m/s) a un chorro fuerte, cruzando el ecuador hacia el noreste, de más de 12 m/s en julio — la firma dinámica exacta del monzón de verano asiático.

### <font color="Navy"> 3.2 Circulación en niveles altos (200 hPa): el anticiclón tibetano <font >

En niveles altos, el verano boreal está dominado por un fuerte **anticiclón** centrado sobre la meseta tibetana (calentada intensamente en verano), con un **chorro tropical del este** (*Tropical Easterly Jet*) en su flanco sur, que se extiende desde el sudeste asiático hasta África. En invierno boreal, en cambio, la región está bajo la influencia del **chorro subtropical del oeste**, mucho más zonal.


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5),
                          subplot_kw={'projection': ccrs.PlateCarree(central_longitude=90)})

cf1 = plot_wind_panel(axes[0], ds, month=1, level=200, extent=ASIA_EXTENT, vmax=45, cmap='plasma')
cf2 = plot_wind_panel(axes[1], ds, month=7, level=200, extent=ASIA_EXTENT, vmax=45, cmap='plasma')

cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', shrink=0.6, pad=0.08)
cbar.set_label('Velocidad del viento en 200 hPa (m/s)')
fig.suptitle('Monzón asiático: viento en niveles altos (200 hPa)', y=1.03, fontsize=13)
plt.show()



Fíjate en el giro anticiclónico centrado cerca del Tíbet en julio (vectores curvándose en sentido horario alrededor de un centro continental), ausente en enero, cuando el flujo es mucho más recto y de oeste a lo largo de todo el dominio (chorro subtropical de invierno).

### <font color="Navy">  3.3 Cizalladura vertical y el índice de Webster–Yang <font >

Webster & Yang (1992) propusieron un índice muy simple pero robusto para medir la intensidad del monzón de verano asiático: **la diferencia de viento zonal entre 850 hPa y 200 hPa, promediada sobre el sur de Asia (0-20°N, 40-110°E)**. Valores muy positivos indican viento del oeste fuerte en niveles bajos junto con viento del este en niveles altos — justo la estructura vertical de un monzón de verano vigoroso.


In [ ]:
def plot_shear_panel(ax, ds, month, extent, vmax=25, cmap='RdBu_r'):
    '''Cizalladura vertical del viento zonal: u(850hPa) - u(200hPa).'''
    u850 = ds.sel(month=month, level=850).u
    u200 = ds.sel(month=month, level=200).u
    shear = u850 - u200

    cf = ax.contourf(ds.longitude, ds.latitude, shear,
                      levels=np.linspace(-vmax, vmax, 21), cmap=cmap,
                      transform=ccrs.PlateCarree(), extend='both')
    ax.coastlines(resolution='110m', linewidth=0.7)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4, color='gray')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 7}
    gl.ylabel_style = {'size': 7}
    ax.set_title(f'Cizalladura u850-u200 — {MONTH_NAMES[month]}', fontsize=10)
    return cf


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5),
                          subplot_kw={'projection': ccrs.PlateCarree(central_longitude=90)})

cf1 = plot_shear_panel(axes[0], ds, month=1, extent=ASIA_EXTENT)
cf2 = plot_shear_panel(axes[1], ds, month=7, extent=ASIA_EXTENT)

# Caja del índice de Webster-Yang
wy_box_lon = [40, 110, 110, 40, 40]
wy_box_lat = [0, 0, 20, 20, 0]
for ax in axes:
    ax.plot(wy_box_lon, wy_box_lat, color='lime', linewidth=2, transform=ccrs.PlateCarree())

cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', shrink=0.6, pad=0.08)
cbar.set_label('u(850hPa) - u(200hPa)  (m/s)')
fig.suptitle('Cizalladura vertical del viento zonal (índice de Webster-Yang)', y=1.03, fontsize=13)
plt.show()


In [ ]:

wy_region = ds.sel(longitude=slice(40, 110), latitude=slice(20, 0))
wy_shear = wy_region.sel(level=850).u - wy_region.sel(level=200).u
wy_index = wy_shear.mean(dim=['longitude', 'latitude'])

for m in [1, 7]:
    print(f"Índice de Webster-Yang ({MONTH_NAMES[m]}): {wy_index.sel(month=m).item():+.2f} m/s")



Con datos reales obtenemos un índice de **≈ −7.7 m/s en enero** frente a **≈ +27.9 m/s en julio**. La literatura considera valores del índice de Webster-Yang por encima de ~15-20 m/s como indicativos de un monzón de verano asiático "fuerte"; nuestro valor de julio (climatología) cae holgadamente en ese régimen, mientras que el signo negativo de enero confirma que en niveles bajos domina entonces un flujo distinto (más débil y/o del este) al de niveles altos — es decir, la estructura vertical característica del **monzón de invierno**, dinámicamente muy distinta a la de verano. Con una serie temporal completa (que obtendrías con el código `cdsapi` de la Sección 2), este mismo índice, calculado mes a mes durante 30-40 años, es una de las métricas estándar para estudiar la variabilidad interanual del monzón indio y su relación con El Niño-Oscilación del Sur (ENSO).


---
<a name='ej-3'></a>
### **<font color="DodgerBlue">Ejercicio: Es su turno de estudiar el monzón de Australia</font>**

## **<font color="DarkBlue"> 4. El monzón australiano </font>**

El monzón australiano (a veces llamado monzón australiano-indonesio) es la contraparte austral del sistema Asia-Australia. Alcanza su apogeo en el **verano austral (diciembre-febrero)**, cuando el norte de Australia y el archipiélago marítimo reciben un flujo húmedo del **noroeste**, en contraste con los vientos alisios del sureste, secos, que dominan en el invierno austral (junio-agosto).

Usaremos un dominio centrado en el norte de Australia y el continente marítimo.

1. Muestra los vientos en 850 hPa en invierno y verano.
2. Muestra los vientos en 200 hPa en invierno y verano.
3. Muestra la cizalladura vertical.
4. Calcula el índice de Webster–Yang. 

Considera la region `AUS_EXTENT = [90, 160, -40, 10]`

---


## <font color= "NavyBlue"> Biblografía complementaria ara consultar: <font>

- Hersbach, H., Bell, B., Berrisford, P., et al. (2020). *The ERA5 global reanalysis*. Quarterly Journal of the Royal Meteorological Society, 146(730), 1999-2049.
- Trenberth, K. E., Stepaniak, D. P., & Caron, J. M. (2000). *The Global Monsoon as Seen through the Divergent Atmospheric Circulation*. Journal of Climate, 13(22), 3969-3993.
- Webster, P. J., & Yang, S. (1992). *Monsoon and ENSO: Selectively Interactive Systems*. Quarterly Journal of the Royal Meteorological Society, 118(507), 877-926.
- Wang, B., & Ding, Q. (2008). *Global monsoon: Dominant mode of annual variation in the tropics*. Dynamics of Atmospheres and Oceans, 44(3-4), 165-183.
- Findlater, J. (1969). *A major low-level air current near the Indian Ocean during the northern summer*. Quarterly Journal of the Royal Meteorological Society, 95(404), 362-380. (Descripción original del chorro de Somalia.)
- Copernicus Climate Change Service (C3S): Climate Data Store, <https://cds.climate.copernicus.eu/>.
